# AIDA 1141 Project Exercise 03 – Data Quality Checks

**Question:** Are the synthetic onboarding request records usable before analysis?

This notebook checks `requests_with_errors.csv` from Exercise 02 for missing values, invalid categories, impossible dates, and duplicates. It documents handling decisions and unresolved issues, then uses the synthetic answer key and original dataset to evaluate detection and repairs.

In [1]:
from pathlib import Path
import pandas as pd

EX02 = Path("../02-synthetic-request-records")
OUTPUT_DIR = Path("outputs")

# Read every value as text so problems are visible exactly as written (nothing auto-converted)
raw = pd.read_csv(EX02 / "data" / "requests_with_errors.csv", dtype=str, keep_default_na=False)
raw.insert(0, "source_row_number", raw.index + 2)  # CSV row number (header = row 1), kept throughout

fields = (pd.read_csv(EX02 / "data-dictionary.csv", dtype=str, keep_default_na=False)
          .query("file == 'requests.csv'").set_index("field"))
FIELD_NAMES = list(fields.index)

# Designed intake issues are counted once per request, so exact duplicate copies are skipped for them
first_copies = raw[~raw.duplicated(subset=FIELD_NAMES, keep="first")]

issues = []  # one record per finding
def add_issue(row, field, check, issue, category):
    issues.append({"source_row_number": row["source_row_number"], "request_id": row["request_id"],
                   "field": field, "check": check, "issue": issue, "category": category,
                   "original_value": row[field] if field else "", "action": "", "resolved": ""})

print(f"Loaded {len(raw):,} rows; columns match the dictionary: {list(raw.columns[1:]) == FIELD_NAMES}")

Loaded 2,072 rows; columns match the dictionary: True


In [2]:
ID_PATTERNS = {"request_id": r"REQ-\d{5}", "employee_id": r"EMP-\d{4}"}
for field, pattern in ID_PATTERNS.items():
    bad = raw[~raw[field].str.fullmatch(pattern)]
    for _, row in bad.iterrows():
        add_issue(row, field, "id_format", f"does not match {pattern}", "data quality")
    print(f"{field}: {len(bad)} values not matching {pattern}")

request_id: 0 values not matching REQ-\d{5}
employee_id: 0 values not matching EMP-\d{4}


In [3]:
MEANING = {
    "requester_comment": ("optional – acceptable", None),
    "submitted_start_date": ("designed intake gap (information issue)", "designed intake issue"),
    "submitted_hiring_department": ("designed intake gap (information issue)", "designed intake issue"),
    "equipment_item": ("missing item on an equipment task (information issue)", "designed intake issue"),
}
summary = []
for field in FIELD_NAMES:
    meaning, category = MEANING.get(field, ("unexpected blank (data quality problem)", "data quality"))
    source = first_copies if category == "designed intake issue" else raw
    blank = source[field] == ""
    if field == "equipment_item":
        # A blank is only expected on equipment tasks; elsewhere not_applicable is expected
        wrong_place = blank & (source["task_type"] != "") & (source["task_type"] != "equipment_setup")
        for _, row in source[wrong_place].iterrows():
            add_issue(row, field, "missing", "blank where not_applicable expected", "data quality")
        blank = blank & ~wrong_place
    if blank.sum():
        summary.append({"field": field, "blanks": int(blank.sum()), "meaning": meaning})
        if category:
            for _, row in source[blank].iterrows():
                add_issue(row, field, "missing", meaning, category)
pd.DataFrame(summary)

,field,blanks,meaning
0,task_type,3,unexpected blank (data quality problem)
1,submitted_hiring_department,80,designed intake gap (information issue)
2,employee_type,2,unexpected blank (data quality problem)
3,intake_channel,3,unexpected blank (data quality problem)
4,submitted_start_date,67,designed intake gap (information issue)
5,equipment_item,4,missing item on an equipment task (information...
6,requester_comment,1514,optional – acceptable


## What the missing values mean

Not every blank is a data-quality error. Missing submitted start dates, hiring departments, and equipment items represent designed intake gaps, while an empty requester comment is acceptable because the field is optional.

For equipment tasks, a blank item means information is missing. For other tasks, `not_applicable` means no equipment item is required. Unexpected blanks in required fields are flagged separately.

In [4]:
def allowed_values(field):
    return [v.strip() for v in fields.loc[field, "allowed_values"].split(";") if v.strip() != "blank"]

for field in fields.index[fields["data_type"].isin(["category", "boolean"])]:
    bad = raw[(raw[field] != "") & ~raw[field].isin(allowed_values(field))]
    for _, row in bad.iterrows():
        add_issue(row, field, "invalid_value", "not in the dictionary's allowed values", "data quality")

# Designed conflict rule (a): an equipment item listed on a non-equipment task
rule_a = first_copies[(first_copies["task_type"] != "") & (first_copies["task_type"] != "equipment_setup")
                      & first_copies["equipment_item"].isin(["laptop", "phone"])]
for _, row in rule_a.iterrows():
    add_issue(row, "equipment_item", "conflict_rule_a", "item listed on a non-equipment task", "designed intake issue")

pd.DataFrame([i for i in issues if i["check"] == "invalid_value"],
             columns=["source_row_number", "field", "original_value"])

,source_row_number,field,original_value
0,319,task_type,orientaton_scheduling
1,1093,task_type,workplace_acess
2,1538,task_type,sytem_access
3,505,responsible_team,Payrol
4,773,responsible_team,I.T.
5,1593,responsible_team,I.T.
6,272,intake_channel,fax
7,737,intake_channel,fax
8,845,intake_channel,fax
9,1132,intake_channel,fax


In [5]:
for field in fields.index[fields["data_type"].isin(["date", "datetime"])]:
    fmt = "%Y-%m-%d" if fields.loc[field, "data_type"] == "date" else "%Y-%m-%dT%H:%M"
    parsed = pd.to_datetime(raw[field], format=fmt, errors="coerce")
    for _, row in raw[(raw[field] != "") & parsed.isna()].iterrows():
        add_issue(row, field, "impossible_date", "not a real calendar date", "data quality")
    if "Weekday" in fields.loc[field, "allowed_values"]:          # explicit dictionary rule
        for _, row in raw[parsed.notna() & (parsed.dt.weekday >= 5)].iterrows():
            add_issue(row, field, "weekend_date", "start date falls on a weekend", "data quality")

# Designed conflict rule (c): an employee's requests give different start dates.
# The data alone cannot show which date is right, so every request involved is flagged for the coordinator.
valid_dates = first_copies[pd.to_datetime(first_copies["submitted_start_date"], format="%Y-%m-%d", errors="coerce").notna()]
n_dates = valid_dates.groupby("employee_id")["submitted_start_date"].nunique()
for _, row in valid_dates[valid_dates["employee_id"].isin(n_dates[n_dates > 1].index)].iterrows():
    add_issue(row, "submitted_start_date", "conflict_rule_c",
              "start dates disagree across this employee's requests", "designed intake issue")

pd.DataFrame([i for i in issues if i["check"] in ("impossible_date", "weekend_date")],
             columns=["source_row_number", "field", "original_value"])

,source_row_number,field,original_value
0,511,submitted_at,2026-02-30T10:15
1,818,submitted_at,2026-04-31T10:15
2,1366,submitted_at,2026-04-31T10:15
3,1009,submitted_start_date,2026-02-30
4,1563,submitted_start_date,2026-06-31


In [6]:
exact_dups = raw[raw.duplicated(subset=FIELD_NAMES, keep="first")]
for _, row in exact_dups.iterrows():
    add_issue(row, "", "duplicate", "exact duplicate of an earlier row", "data quality")

# Same request_id but different values -> flag for review, never auto-keep one
deduped = raw.drop(exact_dups.index)
conflicting = deduped[deduped["request_id"].duplicated(keep=False)]
for _, row in conflicting.iterrows():
    add_issue(row, "request_id", "conflicting_id", "same request_id with different values", "data quality")
print("Exact duplicate copies:", len(exact_dups), "| Rows sharing an ID with different values:", len(conflicting))

Exact duplicate copies: 8 | Rows sharing an ID with different values: 0


## Invalid values, dates, and duplicates

The checks identified 10 invalid categorical values, five impossible dates, and eight exact duplicate copies. No invalid ID formats, weekend start dates, or conflicting request IDs were found.

The intake checks also flagged equipment items listed on non-equipment tasks and disagreements between an employee’s submitted start dates. The date-disagreement check flags all involved requests because it cannot determine which date is correct. It can therefore flag correct dates alongside incorrect ones and would miss dates that all agree but are wrong.

In [7]:
issues_df = pd.DataFrame(issues)
print("Total findings:", len(issues_df))
print(issues_df.groupby(["category", "check"]).size().rename("findings").to_string())
# One request can trigger several checks, so also count unique affected requests
print("\nUnique requests affected:", issues_df.groupby("category")["request_id"].nunique().to_dict())

key = pd.read_csv(EX02 / "data" / "injected_errors.csv", keep_default_na=False)   # used only AFTER detection
dq = issues_df[issues_df["category"] == "data quality"]
found = set(zip(dq["source_row_number"], dq["field"]))
key["detected"] = [(r.source_row_number, r.field) in found for r in key.itertuples()]
planted = set(zip(key["source_row_number"], key["field"]))
print(f"\nPlanted errors detected: {key['detected'].sum()} of {len(key)}")
print("Data-quality findings that were not planted:", len(found - planted))
key.groupby("error_type")["detected"].agg(detected="sum", planted="count")

Total findings: 510
category               check          
data quality           duplicate            8
                       impossible_date      5
                       invalid_value       10
                       missing              8
designed intake issue  conflict_rule_a     60
                       conflict_rule_c    268
                       missing            151

Unique requests affected: {'data quality': 31, 'designed intake issue': 449}

Planted errors detected: 31 of 31
Data-quality findings that were not planted: 0


,detected,planted
error_type,,
duplicate_row,8,8
extra_blank,8,8
impossible_date,5,5
invalid_channel,4,4
misspelled_category,6,6


## Findings and planted-error detection

The checks produced 510 findings: 31 data-quality findings affecting 31 unique requests and 479 designed intake findings affecting 449 unique requests. A request can trigger more than one check, so findings and affected requests are different counts.

The answer key was loaded only after detection. All 31 planted errors were detected, with no additional data-quality findings outside the key. Detecting an error does not mean its original value can be recovered.

In [8]:
cleaned = raw.copy()
def mark(row_number, field, action, resolved):
    m = (issues_df["source_row_number"] == row_number) & (issues_df["field"] == field)
    issues_df.loc[m, ["action", "resolved"]] = [action, resolved]

# 1. Exact duplicates removed; conflicting IDs (none here) would be kept and flagged
for _, row in exact_dups.iterrows():
    mark(row["source_row_number"], "", "removed extra copy", "yes")
cleaned = cleaned.drop(exact_dups.index)

# 2. Documented corrections for the misspellings found in cell D (reviewed by a person)
CORRECTIONS = {
    ("task_type", "orientaton_scheduling"): "orientation_scheduling",
    ("task_type", "workplace_acess"): "workplace_access",
    ("task_type", "sytem_access"): "system_access",
    ("responsible_team", "Payrol"): "Payroll",
    ("responsible_team", "I.T."): "IT",
}
pd.DataFrame([{"field": f, "found": v, "corrected_to": c} for (f, v), c in CORRECTIONS.items()]) \
    .to_csv(OUTPUT_DIR / "correction_mapping.csv", index=False)
for _, issue in issues_df[issues_df["check"] == "invalid_value"].iterrows():
    i = cleaned.index[cleaned["source_row_number"] == issue["source_row_number"]]
    fix = CORRECTIONS.get((issue["field"], issue["original_value"]))
    if fix:
        cleaned.loc[i, issue["field"]] = fix
        mark(issue["source_row_number"], issue["field"], f"corrected to {fix} (documented mapping)", "yes")
    else:
        cleaned.loc[i, issue["field"]] = ""
        mark(issue["source_row_number"], issue["field"], "no valid match; cleaned value left blank", "no")

# 3. Unexpected blanks: recover only from consistent, valid evidence
team_to_task = {"Payroll": "payroll_setup", "Facilities": "workplace_access", "Manager": "orientation_scheduling"}
for _, issue in issues_df[(issues_df["check"] == "missing") & (issues_df["category"] == "data quality")].iterrows():
    i = cleaned.index[cleaned["source_row_number"] == issue["source_row_number"]]
    row = cleaned.loc[i[0]]
    field = issue["field"]
    if field == "employee_type":
        others = cleaned.loc[(cleaned["employee_id"] == row["employee_id"])
                             & cleaned["employee_type"].isin(allowed_values("employee_type")), "employee_type"]
        if others.nunique() == 1:
            cleaned.loc[i, field] = others.iloc[0]
            mark(issue["source_row_number"], field, "recovered: employee's other records agree", "yes")
        else:
            mark(issue["source_row_number"], field, "not recovered: no valid agreeing records", "no")
    elif field == "task_type" and row["responsible_team"] in team_to_task:
        cleaned.loc[i, field] = team_to_task[row["responsible_team"]]
        mark(issue["source_row_number"], field, "recovered: team maps to one task type", "yes")
    else:
        mark(issue["source_row_number"], field, "not recoverable; left blank", "no")

# 4. Impossible dates: submitted_at recovered from the unmodified status event log (treated as a trusted
#    record of the original submission), only when exactly one valid submission event exists
events = pd.read_csv(EX02 / "data" / "status_events.csv", dtype=str, keep_default_na=False)
submissions = events[(events["event_type"] == "submitted")
                     & pd.to_datetime(events["event_at"], format="%Y-%m-%dT%H:%M", errors="coerce").notna()]
for _, issue in issues_df[issues_df["check"] == "impossible_date"].iterrows():
    i = cleaned.index[cleaned["source_row_number"] == issue["source_row_number"]]
    match = submissions.loc[submissions["request_id"] == issue["request_id"], "event_at"]
    if issue["field"] == "submitted_at" and len(match) == 1:
        cleaned.loc[i, "submitted_at"] = match.iloc[0]
        mark(issue["source_row_number"], "submitted_at", "recovered from the single submission event in status_events.csv", "yes")
    else:
        cleaned.loc[i, issue["field"]] = ""
        mark(issue["source_row_number"], issue["field"], "set to blank (no trusted source); confirmed date still available"
             if issue["field"] == "submitted_start_date" else "set to blank (no single trusted source)", "no")

# 5. Any other data-quality finding (e.g., ID format, weekend date, conflicting ID) is kept for a person to review
unhandled = (issues_df["category"] == "data quality") & (issues_df["action"] == "")
issues_df.loc[unhandled, ["action", "resolved"]] = ["retained and flagged for human review", "no"]

# Designed intake issues are expected information gaps, not corruption: kept as recorded
designed = issues_df["category"] == "designed intake issue"
issues_df.loc[designed, ["action", "resolved"]] = ["kept: expected intake issue (resolved later by coordinator)", "n/a"]

# Save outputs
issues_df.to_csv(OUTPUT_DIR / "quality_issues.csv", index=False)
cleaned.drop(columns="source_row_number").to_csv(OUTPUT_DIR / "requests_cleaned.csv", index=False)
quality_report = (issues_df.groupby(["category", "check", "field"])
                  .agg(findings=("issue", "size"),
                       resolved=("resolved", lambda s: (s == "yes").sum()),
                       unresolved=("resolved", lambda s: (s == "no").sum()),
                       handling=("action", lambda s: "; ".join(sorted(set(s)))))
                  .reset_index())
quality_report.to_csv(OUTPUT_DIR / "quality_report.csv", index=False)
quality_report

,category,check,field,findings,resolved,unresolved,handling
0,data quality,duplicate,,8,8,0,removed extra copy
1,data quality,impossible_date,submitted_at,3,3,0,recovered from the single submission event in ...
2,data quality,impossible_date,submitted_start_date,2,0,2,set to blank (no trusted source); confirmed da...
3,data quality,invalid_value,intake_channel,4,0,4,no valid match; cleaned value left blank
4,data quality,invalid_value,responsible_team,3,3,0,corrected to IT (documented mapping); correcte...
5,data quality,invalid_value,task_type,3,3,0,corrected to orientation_scheduling (documente...
6,data quality,missing,employee_type,2,2,0,recovered: employee's other records agree
7,data quality,missing,intake_channel,3,0,3,not recoverable; left blank
8,data quality,missing,task_type,3,2,1,not recoverable; left blank; recovered: team m...
9,designed intake issue,conflict_rule_a,equipment_item,60,0,0,kept: expected intake issue (resolved later by...


## Handling decisions and unresolved issues

Exact duplicate copies were removed, and recognized spelling errors were corrected using the documented mapping. Missing employee types were recovered only when other valid records for that employee agreed. Missing task types were recovered only when the responsible team mapped to one possible task type.

Invalid submission timestamps were recovered from the unmodified status log only when exactly one valid submission event matched. Invalid submitted start dates were left blank because their original values could not be recovered. Unsupported channels were also left blank rather than guessed, with their original values preserved in the issue report.

Designed intake gaps and conflicts were retained as submitted because they describe the workflow rather than file corruption. Any finding without a supported repair rule was retained and flagged for human review.

Of the 31 planted errors, 21 were resolved and 10 remained unresolved: two submitted start dates, four invalid channels, three missing channels, and one missing task type for an IT request.

In [9]:
# Rerun the eight data-quality checks on the cleaned data to confirm independently what remains
# (the designed intake-conflict checks are not repeated: those issues are kept on purpose)
def recheck(df):
    required = [f for f in FIELD_NAMES if f not in MEANING]
    categorical = fields.index[fields["data_type"].isin(["category", "boolean"])]
    def parsed(f):
        fmt = "%Y-%m-%d" if fields.loc[f, "data_type"] == "date" else "%Y-%m-%dT%H:%M"
        return pd.to_datetime(df[f], format=fmt, errors="coerce")
    dates = fields.index[fields["data_type"].isin(["date", "datetime"])]
    weekday_fields = [f for f in dates if "Weekday" in fields.loc[f, "allowed_values"]]
    no_dups = df.drop_duplicates(subset=FIELD_NAMES)
    return {
        "ID format problems": int(sum((~df[f].str.fullmatch(p)).sum() for f, p in ID_PATTERNS.items())),
        "unexpected blanks": int((df[required] == "").sum().sum()),
        "equipment blanks on non-equipment tasks": int(((df["equipment_item"] == "") & (df["task_type"] != "")
                                                       & (df["task_type"] != "equipment_setup")).sum()),
        "invalid values": int(sum(((df[f] != "") & ~df[f].isin(allowed_values(f))).sum() for f in categorical)),
        "impossible dates": int(sum(((df[f] != "") & parsed(f).isna()).sum() for f in dates)),
        "weekend start dates": int(sum((parsed(f).dt.weekday >= 5).sum() for f in weekday_fields)),
        "exact duplicate rows": int(df.duplicated(subset=FIELD_NAMES).sum()),
        "conflicting request IDs": int(no_dups["request_id"].duplicated(keep=False).sum()),
    }

print("Before cleaning:", recheck(raw))
print("After cleaning: ", recheck(cleaned))

# Synthetic-data-only evaluation: compare the repairs with the clean original, matched by request_id
clean_reference = pd.read_csv(EX02 / "data" / "requests.csv", dtype=str, keep_default_na=False)
assert cleaned["request_id"].is_unique and clean_reference["request_id"].is_unique
comparison = cleaned.drop(columns="source_row_number").set_index("request_id").sort_index()
reference = clean_reference.set_index("request_id").sort_index()
assert comparison.index.equals(reference.index), "Request IDs differ"
assert comparison.columns.equals(reference.columns), "Columns differ"
print("\nRequests after cleaning:", len(comparison), "| clean original:", len(reference), "| IDs and columns match")
print("Cells still differing from the clean original:",
      (comparison != reference).sum()[lambda s: s > 0].to_dict())

Before cleaning: {'ID format problems': 0, 'unexpected blanks': 8, 'equipment blanks on non-equipment tasks': 0, 'invalid values': 10, 'impossible dates': 5, 'weekend start dates': 0, 'exact duplicate rows': 8, 'conflicting request IDs': 0}
After cleaning:  {'ID format problems': 0, 'unexpected blanks': 8, 'equipment blanks on non-equipment tasks': 0, 'invalid values': 0, 'impossible dates': 0, 'weekend start dates': 0, 'exact duplicate rows': 0, 'conflicting request IDs': 0}

Requests after cleaning: 2064 | clean original: 2064 | IDs and columns match
Cells still differing from the clean original: {'task_type': 1, 'intake_channel': 7, 'submitted_start_date': 2}


## What the final checks show

The independent recheck repeated eight data-quality checks and found eight remaining unexpected blanks, with zero findings in the other seven categories. There are still 10 unresolved repairs: the other two are impossible submitted start dates that were replaced with blanks. Those blanks are permitted intake gaps, but the original submitted dates remain unknown.

After cleaning, the dataset contains 2,064 requests. Matching records by request ID against the original synthetic dataset confirmed 10 differing cells, corresponding to the unresolved repairs.

Real operational data would not normally have an answer key or an untouched reference dataset. These comparisons support this synthetic exercise only, and the cleaned file is not automatically ready for modelling. Later AIDA 1141 modelling will use the original clean files from Exercise 02.